In [0]:
# =============================================================================
# Notebook 08 — Deep Diagnosis + Two-Stage (Hurdle) Model
# Objective: Investigate why direct regression yields negative R² and implement
#            a two-stage approach:
#              Stage 1: Classifier (active H2 > 0 vs inactive H2 = 0)
#              Stage 2: Regressor on active subset (log1p magnitude)
#            Also run: y-randomization sanity check, learning curves, and
#            per-tier residual analysis.
#
# Key hypothesis: GroupKFold with 25-26 unseen papers per fold + extreme target
#   skew (37 zeros, 56 extreme values) make direct regression fail. Splitting
#   the problem reduces noise and improves signal isolation.
#
# Inputs : ml_perovskites.X_mat_clean  (L1, 72 material features)
#          ml_perovskites.y_targets    (log1p_h2_rate, activity_tier, doi)
#          ml_perovskites.master       (raw categoricals for target encoding)
# Outputs: ml_perovskites.cv_results_stage2  (Delta table)
#          ml_perovskites.cv_results_hurdle   (Delta table)
#          /tmp/08_feature_target_corr.png
#          /tmp/08_learning_curve.png
#          /tmp/08_parity_hurdle.png
#          /tmp/08_yrand.png
# Seed   : 42
# Author : Bernardo Araldi da Silva | UFSC | 2026
# =============================================================================

# =============================================================================
# SECTION 0 — Install dependencies (run this cell alone first)
# =============================================================================
%pip install xgboost --quiet

# =============================================================================
# SECTION 1 — Imports and configuration
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import warnings
from sklearn.model_selection import GroupKFold, learning_curve
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (r2_score, mean_squared_error, mean_absolute_error,
                              classification_report, roc_auc_score)
from sklearn.impute import SimpleImputer
import xgboost as xgb

warnings.filterwarnings("ignore")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
N_SPLITS    = 5
TE_M        = 10.0   # target encoding smoothing factor

print("Section 1 complete.")

# =============================================================================
# SECTION 2 — Load tables
# =============================================================================

print("\n--- Loading tables ---")

y_pd      = spark.table("ml_perovskites.y_targets").toPandas()
xl1_pd    = spark.table("ml_perovskites.X_mat_clean").toPandas()
master_pd = spark.table("ml_perovskites.master").toPandas()

print(f"y_targets  : {y_pd.shape}")
print(f"X_mat_clean: {xl1_pd.shape}")

# =============================================================================
# SECTION 3 — Build aligned arrays
# =============================================================================

print("\n--- Building aligned arrays ---")

META = {"source_row_id", "doi", "base_compound", "family_proposed",
        "formula_reduced", "n_elements", "n_A_elements", "n_B_elements",
        "A_site_elements", "B_site_elements", "parse_ok",
        "Compound_original", "domain"}

CATEGORICAL_COLS_RAW = [
    "cocatalyst_class", "scavenger_class", "run_type",
    "prep_class", "rate_confidence", "cutoff_status",
]

NUMERIC_EXP_COLS_MASTER = [
    "lamp_power_W", "cutoff_nm_final", "catalyst_mass_g",
    "organic_vol_fraction", "thermal_T_C", "thermal_time_h", "volume_mL",
]

feat_cols_l1 = [c for c in xl1_pd.columns if c not in META]

# Build L3 numeric from master (same cols as NB07)
num_exp_raw = master_pd[["source_row_id"] + NUMERIC_EXP_COLS_MASTER].copy()
cat_raw     = master_pd[["source_row_id"] + CATEGORICAL_COLS_RAW].copy()
for col in CATEGORICAL_COLS_RAW:
    cat_raw[col] = (cat_raw[col].fillna("unknown")
                    .astype(str).str.strip().str.lower())

# Align all tables on source_row_id
common_ids = (set(y_pd["source_row_id"])
              & set(xl1_pd["source_row_id"])
              & set(master_pd["source_row_id"]))

y_al   = (y_pd[y_pd["source_row_id"].isin(common_ids)]
          .sort_values("source_row_id").reset_index(drop=True))
xl1_al = (xl1_pd[xl1_pd["source_row_id"].isin(common_ids)]
          .sort_values("source_row_id").reset_index(drop=True))
num_al = (num_exp_raw[num_exp_raw["source_row_id"].isin(common_ids)]
          .sort_values("source_row_id").reset_index(drop=True))
cat_al = (cat_raw[cat_raw["source_row_id"].isin(common_ids)]
          .sort_values("source_row_id").reset_index(drop=True))

y_log    = y_al["log1p_h2_rate"].values.astype(float)
y_binary = (y_al["h2_rate_umol_g_h"].values > 0).astype(int)
groups   = y_al["doi"].values
tiers    = y_al["activity_tier"].values

X_l1     = xl1_al[feat_cols_l1].values.astype(float)
X_l3_num = np.hstack([
    xl1_al[feat_cols_l1].values.astype(float),
    num_al[NUMERIC_EXP_COLS_MASTER].values.astype(float),
])

n_active   = y_binary.sum()
n_inactive = (y_binary == 0).sum()
print(f"Total rows  : {len(y_log)}")
print(f"Active (H2>0): {n_active} ({n_active/len(y_log)*100:.1f}%)")
print(f"Inactive     : {n_inactive} ({n_inactive/len(y_log)*100:.1f}%)")
print(f"X_l1 shape  : {X_l1.shape}")
print(f"X_l3 numeric: {X_l3_num.shape}")

# =============================================================================
# SECTION 4 — Target encoding helper
# =============================================================================

def target_encode_fold(cat_train_df, y_train, cat_val_df, cols, m=10.0):
    """Smoothed target encoding: fit on train, apply to val. No leakage."""
    global_mean = y_train.mean()
    enc_train   = np.zeros((len(cat_train_df), len(cols)), dtype=float)
    enc_val     = np.zeros((len(cat_val_df),   len(cols)), dtype=float)
    for j, col in enumerate(cols):
        train_vals = cat_train_df[col].values
        val_vals   = cat_val_df[col].values
        cat_stats  = {}
        for cat in np.unique(train_vals):
            mask     = train_vals == cat
            n_cat    = mask.sum()
            mean_cat = y_train[mask].mean()
            cat_stats[cat] = (n_cat * mean_cat + m * global_mean) / (n_cat + m)
        enc_train[:, j] = np.array([cat_stats.get(v, global_mean) for v in train_vals])
        enc_val[:, j]   = np.array([cat_stats.get(v, global_mean) for v in val_vals])
    return enc_train, enc_val

# =============================================================================
# SECTION 5 — Diagnostic A: Feature-target correlation
# =============================================================================

print("\n--- Diagnostic A: Feature-target Spearman correlations ---")

from scipy.stats import spearmanr

# Only compute on active subset to avoid zero-inflation masking real correlations
active_mask = y_binary == 1
X_l1_active = X_l1[active_mask]
y_active     = y_log[active_mask]

corrs = []
for i, col in enumerate(feat_cols_l1):
    x_col = X_l1_active[:, i]
    valid  = ~np.isnan(x_col)
    if valid.sum() < 30:
        corrs.append((col, np.nan))
        continue
    rho, pval = spearmanr(x_col[valid], y_active[valid])
    corrs.append((col, rho))

corr_df = (pd.DataFrame(corrs, columns=["feature","spearman_rho"])
           .dropna()
           .sort_values("spearman_rho", key=abs, ascending=False))

print(f"  Top 15 features by |Spearman ρ| (active subset, n={active_mask.sum()}):")
print(corr_df.head(15).to_string(index=False))
print(f"\n  Max |ρ|: {corr_df['spearman_rho'].abs().max():.3f}")
print(f"  Features with |ρ| > 0.15: {(corr_df['spearman_rho'].abs() > 0.15).sum()}")
print(f"  Features with |ρ| > 0.10: {(corr_df['spearman_rho'].abs() > 0.10).sum()}")

# Plot top-30 correlations
fig, ax = plt.subplots(figsize=(10, 7))
top30 = corr_df.head(30)
colors_corr = ["#d62728" if r > 0 else "#1f77b4" for r in top30["spearman_rho"]]
ax.barh(top30["feature"][::-1], top30["spearman_rho"][::-1], color=colors_corr[::-1])
ax.axvline(0, color="k", linewidth=0.8)
ax.axvline( 0.15, color="gray", linestyle="--", linewidth=0.7, label="|ρ|=0.15")
ax.axvline(-0.15, color="gray", linestyle="--", linewidth=0.7)
ax.set_xlabel("Spearman ρ (feature vs log1p H₂ rate, active subset)")
ax.set_title("Feature-target correlation (L1 features, n=active rows)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig("/tmp/08_feature_target_corr.png", dpi=130, bbox_inches="tight")
plt.close()
print("  Saved: /tmp/08_feature_target_corr.png")

# =============================================================================
# SECTION 6 — Diagnostic B: Y-randomization sanity check
# =============================================================================

print("\n--- Diagnostic B: Y-randomization (5 permutations, RF L3_te) ---")
print("  (Permuted R² >> real R² → model is picking up spurious patterns)")

N_PERM = 5
gkf = GroupKFold(n_splits=N_SPLITS)

def cv_r2(X_num, y_target, grps, cat_df, n_splits=N_SPLITS):
    """Run GroupKFold CV with target encoding, return mean R² on log scale."""
    r2s = []
    gkf_inner = GroupKFold(n_splits=n_splits)
    for tr_idx, val_idx in gkf_inner.split(X_num, groups=grps):
        y_tr, y_val = y_target[tr_idx], y_target[val_idx]
        cat_tr = cat_df.iloc[tr_idx][CATEGORICAL_COLS_RAW]
        cat_va = cat_df.iloc[val_idx][CATEGORICAL_COLS_RAW]
        te_tr, te_va = target_encode_fold(cat_tr, y_tr, cat_va,
                                           CATEGORICAL_COLS_RAW, m=TE_M)
        X_tr = np.hstack([X_num[tr_idx], te_tr])
        X_va = np.hstack([X_num[val_idx], te_va])
        pipe = Pipeline([
            ("imp",   SimpleImputer(strategy="median")),
            ("model", RandomForestRegressor(
                n_estimators=200, max_features=0.33, min_samples_leaf=3,
                n_jobs=-1, random_state=RANDOM_SEED)),
        ])
        pipe.fit(X_tr, y_tr)
        r2s.append(r2_score(y_val, pipe.predict(X_va)))
    return np.mean(r2s)

real_r2 = cv_r2(X_l3_num, y_log, groups, cat_al)
perm_r2s = []
for p in range(N_PERM):
    rng     = np.random.default_rng(RANDOM_SEED + p)
    y_perm  = rng.permutation(y_log)
    perm_r2s.append(cv_r2(X_l3_num, y_perm, groups, cat_al))
    print(f"  Perm {p+1}: R²={perm_r2s[-1]:+.3f}")

print(f"\n  Real R²         : {real_r2:+.3f}")
print(f"  Mean permuted R²: {np.mean(perm_r2s):+.3f} ± {np.std(perm_r2s):.3f}")
if real_r2 > np.mean(perm_r2s) + np.std(perm_r2s):
    print("  PASS — real model outperforms permuted baseline.")
else:
    print("  WARNING — real model barely distinguishable from permuted baseline.")

# Y-randomization plot
fig_yr, ax_yr = plt.subplots(figsize=(7, 4))
ax_yr.scatter(range(1, N_PERM+1), perm_r2s, color="#aec6e8", s=80,
              zorder=3, label="Permuted R²")
ax_yr.axhline(real_r2,         color="#d62728", linewidth=1.5, linestyle="-",
              label=f"Real R²={real_r2:+.3f}")
ax_yr.axhline(np.mean(perm_r2s), color="#1f77b4", linewidth=1.0, linestyle="--",
              label=f"Mean perm R²={np.mean(perm_r2s):+.3f}")
ax_yr.set_xlabel("Permutation index")
ax_yr.set_ylabel("CV R² (log scale)")
ax_yr.set_title("Y-randomization: real vs permuted target (RF L3_te)")
ax_yr.legend(fontsize=9)
plt.tight_layout()
plt.savefig("/tmp/08_yrand.png", dpi=130, bbox_inches="tight")
plt.close()
print("  Saved: /tmp/08_yrand.png")

# =============================================================================
# SECTION 7 — Two-stage (hurdle) model
#
# Stage 1: Binary classifier — active (H2 > 0) vs inactive (H2 = 0)
#          Uses L1 material features only (intrinsic property of material)
# Stage 2: Regressor — predict log1p(H2 rate) ONLY for active compounds
#          Uses L3_te (material + experimental + target encoding)
#
# CV protocol: same GroupKFold; Stage 2 trains only on active training rows,
#   predicts on all validation rows (inactive → predict 0, active → predict log).
#   Final prediction: Stage1_pred=0 → y_hat=0, Stage1_pred=1 → Stage2_pred.
# =============================================================================

print("\n--- Two-stage hurdle model ---")

hurdle_results = []
oof_hurdle     = np.full(len(y_log), np.nan)
clf_proba_oof  = np.full(len(y_log), np.nan)

for fold_idx, (train_idx, val_idx) in enumerate(gkf.split(X_l1, groups=groups)):
    y_tr,     y_val     = y_log[train_idx],    y_log[val_idx]
    y_bin_tr, y_bin_val = y_binary[train_idx], y_binary[val_idx]

    # --- Stage 1: classify active vs inactive ---
    clf_pipe = Pipeline([
        ("imp",   SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(
            n_estimators=300, max_features=0.33, min_samples_leaf=2,
            n_jobs=-1, random_state=RANDOM_SEED)),
    ])
    clf_pipe.fit(X_l1[train_idx], y_bin_tr)
    clf_pred_val   = clf_pipe.predict(X_l1[val_idx])
    clf_proba_val  = clf_pipe.predict_proba(X_l1[val_idx])[:, 1]
    clf_proba_oof[val_idx] = clf_proba_val

    # Stage 1 metrics
    n_active_val = y_bin_val.sum()
    n_pred_active = clf_pred_val.sum()

    # --- Stage 2: regress on active training rows only ---
    active_tr_mask = y_bin_tr == 1
    active_val_mask = clf_pred_val == 1   # predicted active in val

    y_pred_final = np.zeros(len(val_idx))   # default: predict 0 (log1p(0)=0)

    if active_tr_mask.sum() >= 10 and active_val_mask.sum() > 0:
        cat_tr_active = cat_al.iloc[train_idx][active_tr_mask][CATEGORICAL_COLS_RAW]
        cat_val_pred_active = cat_al.iloc[val_idx][active_val_mask][CATEGORICAL_COLS_RAW]

        y_tr_active = y_tr[active_tr_mask]
        te_tr_a, te_val_a = target_encode_fold(
            cat_tr_active, y_tr_active, cat_val_pred_active,
            CATEGORICAL_COLS_RAW, m=TE_M
        )

        X_tr_active  = np.hstack([X_l3_num[train_idx][active_tr_mask], te_tr_a])
        X_val_active = np.hstack([X_l3_num[val_idx][active_val_mask],  te_val_a])

        reg_pipe = Pipeline([
            ("imp",   SimpleImputer(strategy="median")),
            ("model", RandomForestRegressor(
                n_estimators=400, max_features=0.33, min_samples_leaf=3,
                n_jobs=-1, random_state=RANDOM_SEED)),
        ])
        reg_pipe.fit(X_tr_active, y_tr_active)
        y_pred_final[active_val_mask] = reg_pipe.predict(X_val_active)

    oof_hurdle[val_idx] = y_pred_final

    # Metrics: evaluate on ALL validation rows
    r2_fold   = r2_score(y_val, y_pred_final)
    rmse_fold = np.sqrt(mean_squared_error(y_val, y_pred_final))
    mae_fold  = mean_absolute_error(y_val, y_pred_final)

    # Also compute R² on active-only validation rows (honest regressor signal)
    active_val_true = y_bin_val == 1
    if active_val_true.sum() > 1:
        r2_active = r2_score(y_val[active_val_true],
                             y_pred_final[active_val_true])
    else:
        r2_active = np.nan

    hurdle_results.append({
        "fold": fold_idx + 1, "model": "RF_hurdle", "layer": "L3_te",
        "r2_log_all": r2_fold, "r2_log_active": r2_active,
        "rmse_log": rmse_fold, "mae_log": mae_fold,
        "n_active_val": int(n_active_val),
        "n_predicted_active": int(n_pred_active),
        "n_val": len(val_idx),
    })

    print(f"  Fold {fold_idx+1}: "
          f"R²(all)={r2_fold:+.3f} | R²(active-only)={r2_active:+.4f} | "
          f"Active in val: true={n_active_val}, pred={n_pred_active}")

hurdle_df = pd.DataFrame(hurdle_results)

print(f"\n  Hurdle model summary:")
print(f"  Mean R²(all)        : {hurdle_df['r2_log_all'].mean():+.3f} "
      f"± {hurdle_df['r2_log_all'].std():.3f}")
print(f"  Mean R²(active-only): {hurdle_df['r2_log_active'].mean():+.3f} "
      f"± {hurdle_df['r2_log_active'].std():.3f}")
print(f"  Mean RMSE (log)     : {hurdle_df['rmse_log'].mean():.3f}")

# =============================================================================
# SECTION 8 — Stage 2 standalone: train/predict only on active rows
#             This is the ceiling for the regression sub-problem.
# =============================================================================

print("\n--- Stage 2 standalone (active rows only, oracle classifier) ---")

active_mask_full = y_binary == 1
X_l3_active  = X_l3_num[active_mask_full]
y_log_active  = y_log[active_mask_full]
groups_active = groups[active_mask_full]
cat_active    = cat_al[active_mask_full].reset_index(drop=True)

stage2_results = []
gkf_active = GroupKFold(n_splits=N_SPLITS)

for fold_idx, (tr_idx, val_idx) in enumerate(
        gkf_active.split(X_l3_active, groups=groups_active)):

    y_tr_a  = y_log_active[tr_idx]
    y_val_a = y_log_active[val_idx]

    cat_tr_a = cat_active.iloc[tr_idx][CATEGORICAL_COLS_RAW]
    cat_va_a = cat_active.iloc[val_idx][CATEGORICAL_COLS_RAW]
    te_tr_a, te_va_a = target_encode_fold(cat_tr_a, y_tr_a, cat_va_a,
                                           CATEGORICAL_COLS_RAW, m=TE_M)

    X_tr_a = np.hstack([X_l3_active[tr_idx], te_tr_a])
    X_va_a = np.hstack([X_l3_active[val_idx], te_va_a])

    for name, pipe in [
        ("RF",      Pipeline([("imp", SimpleImputer(strategy="median")),
                               ("m",  RandomForestRegressor(
                                       n_estimators=400, max_features=0.33,
                                       min_samples_leaf=3, n_jobs=-1,
                                       random_state=RANDOM_SEED))])),
        ("XGBoost", Pipeline([("imp", SimpleImputer(strategy="median")),
                               ("m",  xgb.XGBRegressor(
                                       n_estimators=500, learning_rate=0.04,
                                       max_depth=4, subsample=0.8,
                                       colsample_bytree=0.7, reg_alpha=0.5,
                                       reg_lambda=2.0, random_state=RANDOM_SEED,
                                       verbosity=0, n_jobs=-1))])),
    ]:
        pipe.fit(X_tr_a, y_tr_a)
        y_pred_a = pipe.predict(X_va_a)
        stage2_results.append({
            "fold": fold_idx + 1, "model": name, "layer": "L3_te_active",
            "r2_log":   r2_score(y_val_a, y_pred_a),
            "rmse_log": np.sqrt(mean_squared_error(y_val_a, y_pred_a)),
            "mae_log":  mean_absolute_error(y_val_a, y_pred_a),
            "n_val": len(val_idx),
        })

stage2_df  = pd.DataFrame(stage2_results)
stage2_sum = (stage2_df.groupby("model")
              .agg(r2_mean=("r2_log","mean"), r2_std=("r2_log","std"),
                   rmse=("rmse_log","mean"))
              .reset_index()
              .sort_values("r2_mean", ascending=False))

print(f"\n  Active-only subset: {active_mask_full.sum()} rows, "
      f"{len(np.unique(groups_active))} DOI groups")
print(stage2_sum.to_string(index=False))

# =============================================================================
# SECTION 9 — Parity plot: best hurdle vs Stage 2 standalone
# =============================================================================

print("\n--- Generating parity plots ---")

fig3, axes3 = plt.subplots(1, 2, figsize=(12, 5))

# Hurdle OOF parity
mask_h = ~np.isnan(oof_hurdle)
r2_h   = r2_score(y_log[mask_h], oof_hurdle[mask_h])
tiers_plot = tiers[mask_h]
tier_colors_map = {
    "zero":"#888888", "very_low":"#aec6e8", "low":"#1f77b4",
    "medium":"#2ca02c","high":"#ff7f0e","extreme":"#d62728"
}
c_arr = [tier_colors_map.get(t, "#000") for t in tiers_plot]

sc = axes3[0].scatter(y_log[mask_h], oof_hurdle[mask_h],
                       c=c_arr, alpha=0.4, s=10)
lim0 = [y_log.min()-0.3, y_log.max()+0.3]
axes3[0].plot(lim0, lim0, "k--", linewidth=0.8)
axes3[0].set_xlim(lim0); axes3[0].set_ylim(lim0)
axes3[0].set_xlabel("Actual log1p(H₂ rate)")
axes3[0].set_ylabel("Predicted log1p(H₂ rate)")
axes3[0].set_title(f"OOF Hurdle model (all rows) R²={r2_h:.3f}")

# Add tier legend
from matplotlib.lines import Line2D
handles = [Line2D([0],[0], marker="o", color="w",
           markerfacecolor=v, markersize=7, label=k)
           for k,v in tier_colors_map.items()]
axes3[0].legend(handles=handles, fontsize=7, title="tier")

# Stage 2 standalone OOF for RF (best expected)
# Rebuild OOF for Stage 2
oof_s2 = np.full(active_mask_full.sum(), np.nan)
for fold_idx, (tr_idx, val_idx) in enumerate(
        gkf_active.split(X_l3_active, groups=groups_active)):
    y_tr_a  = y_log_active[tr_idx]
    cat_tr_a = cat_active.iloc[tr_idx][CATEGORICAL_COLS_RAW]
    cat_va_a = cat_active.iloc[val_idx][CATEGORICAL_COLS_RAW]
    te_tr_a, te_va_a = target_encode_fold(cat_tr_a, y_tr_a, cat_va_a,
                                           CATEGORICAL_COLS_RAW, m=TE_M)
    X_tr_a = np.hstack([X_l3_active[tr_idx], te_tr_a])
    X_va_a = np.hstack([X_l3_active[val_idx], te_va_a])
    pipe_s2 = Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("m",   RandomForestRegressor(n_estimators=400, max_features=0.33,
                                      min_samples_leaf=3, n_jobs=-1,
                                      random_state=RANDOM_SEED)),
    ])
    pipe_s2.fit(X_tr_a, y_tr_a)
    oof_s2[val_idx] = pipe_s2.predict(X_va_a)

mask_s2 = ~np.isnan(oof_s2)
r2_s2   = r2_score(y_log_active[mask_s2], oof_s2[mask_s2])

axes3[1].scatter(y_log_active[mask_s2], oof_s2[mask_s2],
                  alpha=0.4, s=10, color="#1f77b4")
lim1 = [y_log_active.min()-0.3, y_log_active.max()+0.3]
axes3[1].plot(lim1, lim1, "k--", linewidth=0.8)
axes3[1].set_xlim(lim1); axes3[1].set_ylim(lim1)
axes3[1].set_xlabel("Actual log1p(H₂ rate) — active rows only")
axes3[1].set_ylabel("Predicted log1p(H₂ rate)")
axes3[1].set_title(f"Stage 2 standalone (active rows, oracle) R²={r2_s2:.3f}")

plt.tight_layout()
plt.savefig("/tmp/08_parity_hurdle.png", dpi=130, bbox_inches="tight")
plt.close()
print("  Saved: /tmp/08_parity_hurdle.png")

# =============================================================================
# SECTION 10 — Learning curve (Stage 2 standalone, RF)
# =============================================================================

print("\n--- Learning curve (Stage 2, RF, active rows) ---")

# Use imputed X only (no target encoding for LC, to keep it simple)
imputer_lc = SimpleImputer(strategy="median")
X_lc = imputer_lc.fit_transform(X_l3_active)

rf_lc = RandomForestRegressor(n_estimators=200, max_features=0.33,
                               min_samples_leaf=3, n_jobs=-1,
                               random_state=RANDOM_SEED)

train_sizes_rel = np.linspace(0.10, 1.0, 8)
train_sizes, train_scores, val_scores = learning_curve(
    rf_lc, X_lc, y_log_active,
    cv=GroupKFold(n_splits=N_SPLITS),
    groups=groups_active,
    train_sizes=train_sizes_rel,
    scoring="r2",
    n_jobs=-1,
)

fig_lc, ax_lc = plt.subplots(figsize=(7, 4))
ax_lc.plot(train_sizes, train_scores.mean(axis=1), "o-",
           color="#d62728", label="Train R²")
ax_lc.fill_between(train_sizes,
                    train_scores.mean(axis=1) - train_scores.std(axis=1),
                    train_scores.mean(axis=1) + train_scores.std(axis=1),
                    alpha=0.2, color="#d62728")
ax_lc.plot(train_sizes, val_scores.mean(axis=1), "o-",
           color="#1f77b4", label="CV R²")
ax_lc.fill_between(train_sizes,
                    val_scores.mean(axis=1) - val_scores.std(axis=1),
                    val_scores.mean(axis=1) + val_scores.std(axis=1),
                    alpha=0.2, color="#1f77b4")
ax_lc.axhline(0.58, color="black", linestyle="--", linewidth=0.9,
              label="ICC ceiling (0.58)")
ax_lc.set_xlabel("Training set size (active rows)")
ax_lc.set_ylabel("R² (log1p scale)")
ax_lc.set_title("Learning curve — RF Stage 2 (active rows, GroupKFold)")
ax_lc.legend(fontsize=9)
ax_lc.set_ylim(bottom=-0.5, top=1.0)
plt.tight_layout()
plt.savefig("/tmp/08_learning_curve.png", dpi=130, bbox_inches="tight")
plt.close()
print("  Saved: /tmp/08_learning_curve.png")

# =============================================================================
# SECTION 11 — Write results to Delta tables
# =============================================================================

print("\n--- Writing results to Delta tables ---")

s2_spark = spark.createDataFrame(
    stage2_df.where(pd.notnull(stage2_df), other=None)
)
(s2_spark.write.format("delta").mode("overwrite")
 .option("overwriteSchema","true")
 .saveAsTable("ml_perovskites.cv_results_stage2"))
n_s2 = spark.table("ml_perovskites.cv_results_stage2").count()
print(f"  cv_results_stage2 : {n_s2} rows")

hurdle_spark = spark.createDataFrame(
    hurdle_df.where(pd.notnull(hurdle_df), other=None)
)
(hurdle_spark.write.format("delta").mode("overwrite")
 .option("overwriteSchema","true")
 .saveAsTable("ml_perovskites.cv_results_hurdle"))
n_h = spark.table("ml_perovskites.cv_results_hurdle").count()
print(f"  cv_results_hurdle : {n_h} rows")

# =============================================================================
# SECTION 12 — Final summary
# =============================================================================

print("\n" + "="*60)
print("NOTEBOOK 08 COMPLETE")
print("="*60)

print(f"\n[A] Feature-target Spearman ρ (active subset):")
print(f"    Max |ρ|                  : {corr_df['spearman_rho'].abs().max():.3f}")
print(f"    Features with |ρ| > 0.15 : {(corr_df['spearman_rho'].abs() > 0.15).sum()}")
print(f"    Features with |ρ| > 0.10 : {(corr_df['spearman_rho'].abs() > 0.10).sum()}")

print(f"\n[B] Y-randomization (RF L3_te):")
print(f"    Real R²          : {real_r2:+.3f}")
print(f"    Mean permuted R² : {np.mean(perm_r2s):+.3f}")

print(f"\n[C] Two-stage hurdle (Stage1=RF classifier, Stage2=RF regressor):")
print(f"    R²(all rows)     : {hurdle_df['r2_log_all'].mean():+.3f}")
print(f"    R²(active-only)  : {hurdle_df['r2_log_active'].mean():+.3f}")

print(f"\n[D] Stage 2 standalone (active rows only, oracle classifier):")
for _, row in stage2_sum.iterrows():
    print(f"    {row['model']:8s}  R²={row['r2_mean']:+.3f}±{row['r2_std']:.3f}  "
          f"RMSE={row['rmse']:.3f}")

print(f"\nICC(DOI) ceiling : 0.58")
s2_best = stage2_sum.iloc[0]
gap = 0.58 - s2_best["r2_mean"]
print(f"Gap to ceiling   : {gap:.3f} R² units (best={s2_best['model']} "
      f"R²={s2_best['r2_mean']:+.3f})")

print(f"\nOutput tables:")
print(f"  ml_perovskites.cv_results_stage2 : {n_s2} rows")
print(f"  ml_perovskites.cv_results_hurdle : {n_h} rows")

print(f"\nNext step:")
s2_best_r2 = s2_best["r2_mean"]
if s2_best_r2 > 0.35:
    print(f"  Stage 2 R² > 0.35 — good signal. Proceed to NB09: Optuna HPO")
    print(f"  on Stage 2 regressor (RF + XGBoost, active rows, L3_te).")
elif s2_best_r2 > 0.10:
    print(f"  Stage 2 R² positive but modest. Proceed to NB09: Optuna HPO")
    print(f"  AND add L2 thermodynamic features (thermo library).")
else:
    print(f"  Stage 2 R² < 0.10 — check learning curve and correlation plot.")
    print(f"  If learning curve still flat → dataset too small for active subset.")
    print(f"  Consider: expand dataset, add DFT features, or change strategy.")